In [1]:
from IPython.display import HTML
HTML('''<style>.jp-Cell-inputWrapper, .input { margin-top: 0.5em; }</style>''')

# Notebook 00 — Data Acquisition
## ENGG2112 Project MODR

This notebook acquires raw flu surveillance + demographic data from each viable source. It outputs **tidy per-source CSVs** in standardised formats. No merging or feature engineering happens here — that's `01_master_dataframe.ipynb`.

### Actual viable sources (after exhaustive research)
We tested ~25 US states for county-level flu data via API. After verifying claims by hitting actual endpoints, only four states publish what we need:

| State | Areas | Format | Source |
|---|---|---|---|
| **New York** | 62 counties | Weekly lab-confirmed cases by county | NY DOH Open Data |
| **Pennsylvania** | 67 counties | Cumulative cases per season by county | PA Open Data Socrata API |
| **Connecticut** | 9 planning regions | Individual case records (post-2022 boundary) | CT Open Data Socrata API |
| **Delaware** | 3 counties | Weekly new cases by indicator | DE Open Data Socrata API |

Combined: **141 areas** for cross-sectional analysis. CA, FL, VA, CO, RI, NJ, MD, IA, UT, ME, VT, NH, NM all dead ends — see Appendix.

### Metrics computed per (area, season)
- `total_cases` — outbreak burden
- `weeks_active` — outbreak duration
- `peak_week_cases` — outbreak intensity
- `time_to_peak` — weeks from season start until peak (outbreak speed)
- `outbreak_steepness` — peak / time_to_peak (climb rate)
- `pct_type_a` — strain composition (NY/CT only)

### Outputs
| File | Description |
|---|---|
| `ny_flu_panel.csv` | NY: (fips, season) panel, 17 seasons |
| `pennsylvania/pa_flu_panel.csv` | PA: cumulative 2025-26 |
| `connecticut/ct_flu_panel.csv` | CT: 9 planning regions × 3 seasons |
| `delaware/de_flu_panel.csv` | DE: 3 counties × 6 seasons |
| `acs_demographics/acs_multistate_2022.csv` | ACS demographics for 141 areas |
| `acs_demographics/land_area_2022.csv` | Land area for density calc |

## Setup

In [2]:
import pandas as pd
import numpy as np
import urllib.request
import urllib.parse
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'usa_data'
RAW_DIR.mkdir(parents=True, exist_ok=True)
for sub in ['pennsylvania', 'connecticut', 'delaware', 'acs_demographics']:
    (RAW_DIR / sub).mkdir(exist_ok=True)

env_path = PROJECT_ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line.strip() and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()

CENSUS_API_KEY = os.environ.get('CENSUS_API_KEY')
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DIR}')
print(f'Census API key loaded: {"✅" if CENSUS_API_KEY else "❌ MISSING"}')

Project root: /Users/rocbarraket/Documents/ENGG2112/ENGG2112_Project
Raw data dir: /Users/rocbarraket/Documents/ENGG2112/ENGG2112_Project/data/raw/usa_data
Census API key loaded: ✅


---

# 1. New York State

## 1.1 NY Flu — Temporal Panel with Outbreak Dynamics

### Source
- *"Influenza Laboratory-Confirmed Cases by County, Beginning 2009-10 Season"*
- NY State Department of Health: [health.data.ny.gov](https://health.data.ny.gov/)

### Per-(fips, season) metrics
| Column | Description |
|---|---|
| `total_cases` | Sum across all weeks/types |
| `weeks_active` | Distinct CDC weeks reported |
| `peak_week_cases` | Max single-week count (intensity) |
| `time_to_peak` | Weeks until peak (outbreak speed) |
| `outbreak_steepness` | Peak / time_to_peak (climb rate) |
| `pct_type_a` | % Influenza A |

In [3]:
ny_flu_raw = pd.read_csv(
    RAW_DIR / 'Influenza_Laboratory-Confirmed_Cases_by_County__Beginning_2009-10_Season_20260424.csv',
    encoding='utf-8-sig'
)
ny_flu_raw.columns = ny_flu_raw.columns.str.strip().str.lower().str.replace(' ', '_')
ny_flu_raw['count'] = ny_flu_raw['count'].astype(str).str.replace(',', '').astype(int)
ny_flu_raw['fips'] = ny_flu_raw['fips'].astype(str).str.zfill(5)
ny_flu_raw['week_ending_date'] = pd.to_datetime(ny_flu_raw['week_ending_date'], format='%m/%d/%Y')

print(f'Raw NY: {len(ny_flu_raw):,} rows, {ny_flu_raw["county"].nunique()} counties, {ny_flu_raw["season"].nunique()} seasons')

Raw NY: 101,532 rows, 62 counties, 17 seasons


In [4]:
def compute_panel_metrics(group):
    g = group.sort_values('week_ending_date')
    weekly = g.groupby('week_ending_date')['count'].sum().sort_index()
    total = int(weekly.sum())
    weeks_active = int((weekly > 0).sum())
    peak = int(weekly.max()) if len(weekly) else 0
    if total > 0:
        peak_idx = int(weekly.values.argmax())
        time_to_peak = peak_idx + 1
        steepness = round(peak / time_to_peak, 2)
    else:
        time_to_peak = np.nan
        steepness = np.nan
    return pd.Series({'total_cases': total, 'weeks_active': weeks_active,
                      'peak_week_cases': peak, 'time_to_peak': time_to_peak,
                      'outbreak_steepness': steepness})

ny_panel = ny_flu_raw.groupby(['fips', 'county', 'season']).apply(
    compute_panel_metrics, include_groups=False).reset_index()

type_a = (ny_flu_raw[ny_flu_raw['disease'] == 'INFLUENZA_A']
          .groupby(['fips', 'season'])['count'].sum().reset_index()
          .rename(columns={'count': 'type_a_cases'}))
ny_panel = ny_panel.merge(type_a, on=['fips', 'season'], how='left')
ny_panel['type_a_cases'] = ny_panel['type_a_cases'].fillna(0)
ny_panel['pct_type_a'] = (ny_panel['type_a_cases'] / ny_panel['total_cases'].replace(0, np.nan) * 100).round(1)
ny_panel = ny_panel.drop(columns='type_a_cases')

ny_panel['state'] = 'NY'
ny_panel['state_fips'] = '36'
ny_panel['season_start_year'] = ny_panel['season'].str.split('-').str[0].astype(int)
ny_panel = ny_panel[['state', 'state_fips', 'fips', 'county', 'season', 'season_start_year',
                     'total_cases', 'weeks_active', 'peak_week_cases',
                     'time_to_peak', 'outbreak_steepness', 'pct_type_a']]
ny_panel = ny_panel.sort_values(['fips', 'season_start_year']).reset_index(drop=True)
ny_panel.to_csv(RAW_DIR / 'ny_flu_panel.csv', index=False)
print(f'✅ NY panel: {ny_panel.shape}, {ny_panel["fips"].nunique()} counties × {ny_panel["season"].nunique()} seasons')

✅ NY panel: (1032, 12), 62 counties × 17 seasons


## 1.2 NY Demographics — Tidy Snapshot

Validates and saves the existing NY ACS snapshot, flagging the misnamed `elderly_in_group_quarters` column (it's actually total elderly, confirmed bug).

In [5]:
ny_demo_raw = pd.read_csv(RAW_DIR / 'ny_county_demographics.csv')
ny_demo_raw['fips'] = ny_demo_raw['fips'].astype(str).str.zfill(5)
elderly_cols = [c for c in ny_demo_raw.columns if c.startswith(('male_6', 'male_7', 'male_8', 'female_6', 'female_7', 'female_8'))]
diff = (ny_demo_raw[elderly_cols].sum(axis=1) - ny_demo_raw['elderly_in_group_quarters']).abs().sum()
assert diff == 0
ny_demo_raw = ny_demo_raw.rename(columns={'elderly_in_group_quarters': 'elderly_total_RAW_misnamed'})
ny_demo_raw['state'] = 'NY'
ny_demo_raw.to_csv(RAW_DIR / 'ny_demographics_tidy.csv', index=False)
print(f'✅ NY demographics: {ny_demo_raw.shape}')

✅ NY demographics: (62, 64)


## 1.3 NY Vaccination (2024-25 only — supplementary)

In [6]:
ny_vax_raw = pd.read_csv(RAW_DIR / 'New_York_State_COVID-19_and_Influenza_Vaccination_Data_20260424.csv')
ny_vax_raw.columns = ny_vax_raw.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')
ny_vax_county = ny_vax_raw[ny_vax_raw['geography_level'] == 'COUNTY'].copy()
for col in ['covid_19_dose_count', 'influenza_dose_count']:
    if col in ny_vax_county.columns:
        ny_vax_county[col] = ny_vax_county[col].astype(str).str.replace(',', '').replace('nan', '0').astype(int)
ny_vax_county['week_ending'] = pd.to_datetime(ny_vax_county['week_ending'])
ny_vax_county.to_csv(RAW_DIR / 'ny_vaccination_2024.csv', index=False)
print(f'✅ NY vaccination: {ny_vax_county.shape}, total flu doses: {ny_vax_county["influenza_dose_count"].sum():,}')

✅ NY vaccination: (2542, 6), total flu doses: 3,058,740


---

# 2. Pennsylvania

## 2.1 PA Flu — Cumulative 2025-26 Season

### Source
- PA Open Data: *"Cumulative Influenza and RSV Case Counts by County 2025-2026"*
- Endpoint: `https://data.pa.gov/resource/mrpb-ugjv.csv`
- 67 counties, FIPS state code 42, current season only

### Limitation
PA only publishes the current season. We use 2025-26 as PA's contribution — temporally aligned with NY 2025-26.

In [7]:
pa_url = 'https://data.pa.gov/resource/mrpb-ugjv.csv?$limit=1000'
pa_raw_path = RAW_DIR / 'pennsylvania' / 'pa_cumulative_2025-26.csv'
urllib.request.urlretrieve(pa_url, pa_raw_path)
pa_raw = pd.read_csv(pa_raw_path)
print(f'PA raw: {pa_raw.shape}')

# Build PA county FIPS map from Census API
pa_fips_url = f'https://api.census.gov/data/2022/acs/acs5?get=NAME&for=county:*&in=state:42&key={CENSUS_API_KEY}'
with urllib.request.urlopen(pa_fips_url) as r:
    pa_fips_data = json.loads(r.read())
pa_fips_df = pd.DataFrame(pa_fips_data[1:], columns=pa_fips_data[0])
pa_fips_df['fips'] = pa_fips_df['state'] + pa_fips_df['county']
pa_fips_df['county_clean'] = pa_fips_df['NAME'].str.replace(' County, Pennsylvania', '', regex=False).str.upper()
pa_fips_map = dict(zip(pa_fips_df['county_clean'], pa_fips_df['fips']))

# Standardise PA panel
pa_panel = pa_raw.copy()
pa_panel['county_clean'] = pa_panel['county'].str.upper().str.strip()
pa_panel['fips'] = pa_panel['county_clean'].map(pa_fips_map)
assert pa_panel['fips'].notna().all(), f"Unmapped: {pa_panel[pa_panel['fips'].isna()]['county'].tolist()}"

pa_panel = pa_panel.rename(columns={'influenza_cases': 'total_cases', 'county_clean': 'county'})
for col in ['weeks_active', 'peak_week_cases', 'time_to_peak', 'outbreak_steepness', 'pct_type_a']:
    pa_panel[col] = np.nan
pa_panel['season'] = '2025-2026'
pa_panel['season_start_year'] = 2025
pa_panel['state'] = 'PA'
pa_panel['state_fips'] = '42'
pa_panel = pa_panel[['state', 'state_fips', 'fips', 'county', 'season', 'season_start_year',
                     'total_cases', 'weeks_active', 'peak_week_cases',
                     'time_to_peak', 'outbreak_steepness', 'pct_type_a']]
pa_panel.to_csv(RAW_DIR / 'pennsylvania' / 'pa_flu_panel.csv', index=False)
print(f'✅ PA panel: {pa_panel.shape}, total flu cases: {pa_panel["total_cases"].sum():,}')

PA raw: (67, 5)


✅ PA panel: (67, 13), total flu cases: 143,092


---

# 3. Connecticut

## 3.1 CT Flu — Multi-Season Case-Level Data

### Source
- CT Open Data: *"Connecticut Reportable Disease Case List with the County of Residence"*
- Endpoint: `https://data.ct.gov/resource/4rss-apm8.csv?disease=Influenza`
- One row per individual case
- 9 Planning Regions × 3 seasons (2022-23, 2023-24, 2024-25)

### ⚠️ Counties → Planning Regions
CT switched from 8 counties to 9 Planning Regions in 2022 for Census purposes. We aggregate flu data by `incits` (planning region FIPS 09110-09190) to match ACS 2022.

In [8]:
ct_url = 'https://data.ct.gov/resource/4rss-apm8.csv?disease=Influenza&$limit=200000'
ct_raw_path = RAW_DIR / 'connecticut' / 'ct_flu_cases_raw.csv'
urllib.request.urlretrieve(ct_url, ct_raw_path)
ct_raw = pd.read_csv(ct_raw_path)
print(f'CT raw: {ct_raw.shape}')

# Aggregate by planning region
ct = ct_raw[ct_raw['incits'].notna() & ct_raw['temporalityvalue'].notna()].copy()
ct['fips'] = ct['incits'].astype(int).astype(str).str.zfill(5)

ct_agg = (ct.groupby(['fips', 'cepr', 'temporalityvalue']).size()
          .reset_index(name='total_cases')
          .rename(columns={'temporalityvalue': 'season', 'cepr': 'county'}))

ct['report_week'] = pd.to_datetime(ct['reportperiodstart']).dt.to_period('W').astype(str)
weekly = ct.groupby(['fips', 'temporalityvalue', 'report_week']).size().reset_index(name='week_cases')

ct_dyn = (weekly.groupby(['fips', 'temporalityvalue'])
          .agg(weeks_active=('week_cases', lambda s: int((s > 0).sum())),
               peak_week_cases=('week_cases', 'max')).reset_index()
          .rename(columns={'temporalityvalue': 'season'}))
ct_agg = ct_agg.merge(ct_dyn, on=['fips', 'season'], how='left')

def ct_dyn_grp(g):
    g = g.sort_values('report_week')
    w = g['week_cases'].values
    if len(w) == 0 or w.max() == 0:
        return pd.Series({'time_to_peak': np.nan, 'outbreak_steepness': np.nan})
    ttp = int(w.argmax()) + 1
    return pd.Series({'time_to_peak': ttp, 'outbreak_steepness': round(w.max() / ttp, 2)})

ct_extra = (weekly.groupby(['fips', 'temporalityvalue']).apply(ct_dyn_grp, include_groups=False)
            .reset_index().rename(columns={'temporalityvalue': 'season'}))
ct_agg = ct_agg.merge(ct_extra, on=['fips', 'season'], how='left')

ct_a = (ct[ct['descriptorvalue'].fillna('').str.startswith('A')]
        .groupby(['fips', 'temporalityvalue']).size().reset_index(name='ta')
        .rename(columns={'temporalityvalue': 'season'}))
ct_agg = ct_agg.merge(ct_a, on=['fips', 'season'], how='left')
ct_agg['ta'] = ct_agg['ta'].fillna(0)
ct_agg['pct_type_a'] = (ct_agg['ta'] / ct_agg['total_cases'].replace(0, np.nan) * 100).round(1)
ct_agg = ct_agg.drop(columns='ta')

ct_agg['state'] = 'CT'
ct_agg['state_fips'] = '09'
ct_agg['season_start_year'] = ct_agg['season'].str.split('-').str[0].astype(int)
ct_agg['county'] = ct_agg['county'].str.replace(' Planning Region', '', regex=False).str.upper()
ct_agg = ct_agg[['state', 'state_fips', 'fips', 'county', 'season', 'season_start_year',
                 'total_cases', 'weeks_active', 'peak_week_cases',
                 'time_to_peak', 'outbreak_steepness', 'pct_type_a']]
ct_agg = ct_agg.sort_values(['fips', 'season_start_year']).reset_index(drop=True)
ct_agg.to_csv(RAW_DIR / 'connecticut' / 'ct_flu_panel.csv', index=False)
print(f'✅ CT panel: {ct_agg.shape}, {ct_agg["fips"].nunique()} planning regions × {ct_agg["season"].nunique()} seasons')

CT raw: (119192, 17)
✅ CT panel: (27, 12), 9 planning regions × 3 seasons


---

# 4. Delaware

## 4.1 DE Flu — Multi-Season Weekly Cases

### Source
- DE Open Data: *"Delaware Influenza Cases"*
- Endpoint: `https://data.delaware.gov/resource/46y5-s57v.csv`
- 3 counties × 6+ seasons (2019-20 to 2025-26 in progress)
- Indicator: `New Flu Cases` (weekly increments) at county level

### Method
1. Filter to county-level rows (location ∈ {'Kent County', 'New Castle County', 'Sussex County'})
2. Filter to indicator='New Flu Cases'
3. Bucket weeks into flu seasons (Oct year N → May year N+1)
4. Compute the same metrics as NY

### FIPS codes
- Kent County: 10001
- New Castle County: 10003
- Sussex County: 10005

In [9]:
# Download Delaware flu data
de_url = 'https://data.delaware.gov/resource/46y5-s57v.csv?$limit=500000'
de_raw_path = RAW_DIR / 'delaware' / 'de_flu_cases_raw.csv'
urllib.request.urlretrieve(de_url, de_raw_path)
de_raw = pd.read_csv(de_raw_path)
print(f'DE raw: {de_raw.shape}')

# Filter to county-level rows (where location is a county name) and to "New Flu Cases" indicator
de = de_raw[de_raw['location'].isin(['Kent County', 'New Castle County', 'Sussex County'])].copy()
de = de[de['indicator'] == 'New Flu Cases'].copy()  # Weekly new cases (not cumulative)
print(f'After filter: {de.shape}')

de['start_date'] = pd.to_datetime(de['start_date'])
de['end_date'] = pd.to_datetime(de['end_date'])
de['period_days'] = (de['end_date'] - de['start_date']).dt.days

# Map locations to FIPS
de_fips_map = {'Kent County': '10001', 'New Castle County': '10003', 'Sussex County': '10005'}
de['fips'] = de['location'].map(de_fips_map)

# Assign each row to a flu season based on start_date
# Flu season: Oct year N to May year N+1 → labelled "N-(N+1)"
def flu_season(date):
    yr = date.year
    if date.month >= 9:  # Sept onwards = next season
        return f'{yr}-{yr+1}'
    else:  # before Sept = current season
        return f'{yr-1}-{yr}'

de['season'] = de['start_date'].apply(flu_season)
de['season_start_year'] = de['season'].str.split('-').str[0].astype(int)

# Aggregate weekly DE data to (fips, season) panel
de_panel = (de.groupby(['fips', 'location', 'season', 'season_start_year'])
            .agg(total_cases=('value', 'sum'),
                 weeks_active=('value', lambda s: int((s > 0).sum())),
                 peak_week_cases=('value', 'max'))
            .reset_index()
            .rename(columns={'location': 'county'}))

# time_to_peak / steepness
def de_dyn(grp):
    g = grp.sort_values('start_date')
    vals = g['value'].values
    if len(vals) == 0 or vals.max() == 0:
        return pd.Series({'time_to_peak': np.nan, 'outbreak_steepness': np.nan})
    ttp = int(vals.argmax()) + 1
    return pd.Series({'time_to_peak': ttp, 'outbreak_steepness': round(vals.max() / ttp, 2)})

de_extra = (de.groupby(['fips', 'season']).apply(de_dyn, include_groups=False).reset_index())
de_panel = de_panel.merge(de_extra, on=['fips', 'season'], how='left')

# DE doesn't publish flu type A/B breakdown at county level — leave as NaN
de_panel['pct_type_a'] = np.nan

# Add metadata
de_panel['state'] = 'DE'
de_panel['state_fips'] = '10'
de_panel['county'] = de_panel['county'].str.replace(' County', '', regex=False).str.upper()

# Drop incomplete seasons (less than 4 weeks of data — likely fragments at boundaries)
de_panel = de_panel[de_panel['weeks_active'] >= 4].copy()

de_panel = de_panel[['state', 'state_fips', 'fips', 'county', 'season', 'season_start_year',
                     'total_cases', 'weeks_active', 'peak_week_cases',
                     'time_to_peak', 'outbreak_steepness', 'pct_type_a']]
de_panel = de_panel.sort_values(['fips', 'season_start_year']).reset_index(drop=True)
de_panel.to_csv(RAW_DIR / 'delaware' / 'de_flu_panel.csv', index=False)

print(f'\n✅ DE panel: {de_panel.shape}, {de_panel["fips"].nunique()} counties × {de_panel["season"].nunique()} seasons')
print(f'\nDE panel sample (Kent, all seasons):')
print(de_panel[de_panel["fips"] == "10001"].to_string(index=False))

DE raw: (500000, 9)
After filter: (726, 9)

✅ DE panel: (18, 12), 3 counties × 6 seasons

DE panel sample (Kent, all seasons):
state state_fips  fips county    season  season_start_year  total_cases  weeks_active  peak_week_cases  time_to_peak  outbreak_steepness  pct_type_a
   DE         10 10001   KENT 2019-2020               2019       1819.0            21            184.0          14.0               13.14         NaN
   DE         10 10001   KENT 2021-2022               2021        622.0            32             79.0          26.0                3.04         NaN
   DE         10 10001   KENT 2022-2023               2022       4475.0            38           1261.0          13.0               97.00         NaN
   DE         10 10001   KENT 2023-2024               2023       4313.0            39            547.0          27.0               20.26         NaN
   DE         10 10001   KENT 2024-2025               2024       6220.0            38           1337.0          28.0            

---

# 5. Multi-State ACS Demographics

Pull the same demographic variables for all NY + PA + CT + DE areas from the US Census ACS 2022 5-year API.

### Variables
~12 core features mapped to ACS variable codes:

| Variable | ACS Code |
|---|---|
| Total population | B01003_001E |
| Median age | B01002_001E |
| Population 65+ | B01001_020-025 + 044-049 |
| Median income | B19013_001E |
| Below poverty | B17001_002E |
| Civilian labour force | B23025_003E |
| Unemployed | B23025_005E |
| Total commuters | B08301_001E |
| Public transport | B08301_010E |
| Bachelor's+ | B15003_022-025 |
| White population | B02001_002E |
| Foreign-born | B05002_013E |
| Average household size | B25010_001E |

In [10]:
ACS_VARS = {
    'pop_total': 'B01003_001E',
    'median_age': 'B01002_001E',
    'median_income': 'B19013_001E',
    'pop_below_poverty': 'B17001_002E',
    'labour_force': 'B23025_003E',
    'unemployed': 'B23025_005E',
    'total_commuters': 'B08301_001E',
    'public_transport_commuters': 'B08301_010E',
    'pop_white': 'B02001_002E',
    'pop_foreign_born': 'B05002_013E',
    'avg_household_size': 'B25010_001E',
}
ELDERLY_M = [f'B01001_{i:03d}E' for i in range(20, 26)]
ELDERLY_F = [f'B01001_{i:03d}E' for i in range(44, 50)]
BACH_PLUS = [f'B15003_{i:03d}E' for i in [22, 23, 24, 25]]
ALL_VARS = list(ACS_VARS.values()) + ELDERLY_M + ELDERLY_F + BACH_PLUS
print(f'ACS variables to fetch: {len(ALL_VARS)}')

ACS variables to fetch: 27


In [11]:
def fetch_acs(state_fips, year=2022):
    var_list = ','.join(['NAME'] + ALL_VARS)
    url = (f'https://api.census.gov/data/{year}/acs/acs5'
           f'?get={var_list}&for=county:*&in=state:{state_fips}&key={CENSUS_API_KEY}')
    with urllib.request.urlopen(url, timeout=30) as r:
        data = json.loads(r.read())
    df = pd.DataFrame(data[1:], columns=data[0])
    df['fips'] = df['state'] + df['county']
    return df

print('Fetching ACS data...')
acs_ny = fetch_acs('36'); print(f'  NY: {len(acs_ny)} counties')
acs_pa = fetch_acs('42'); print(f'  PA: {len(acs_pa)} counties')
acs_ct = fetch_acs('09'); print(f'  CT: {len(acs_ct)} planning regions')
acs_de = fetch_acs('10'); print(f'  DE: {len(acs_de)} counties')

acs_combined = pd.concat([acs_ny, acs_pa, acs_ct, acs_de], ignore_index=True)
print(f'\nCombined: {acs_combined.shape}, total areas: {acs_combined["fips"].nunique()}')

Fetching ACS data...


  NY: 62 counties


  PA: 67 counties


  CT: 9 planning regions


  DE: 3 counties

Combined: (141, 31), total areas: 141


In [12]:
numeric_cols = [c for c in acs_combined.columns if c.startswith('B')]
for c in numeric_cols:
    acs_combined[c] = pd.to_numeric(acs_combined[c], errors='coerce')

acs_combined['pop_elderly'] = acs_combined[ELDERLY_M + ELDERLY_F].sum(axis=1)
acs_combined['pop_bachelors_plus'] = acs_combined[BACH_PLUS].sum(axis=1)

rename_map = {v: k for k, v in ACS_VARS.items()}
acs_combined = acs_combined.rename(columns=rename_map)
acs_combined = acs_combined.drop(columns=ELDERLY_M + ELDERLY_F + BACH_PLUS)

state_map = {'36': 'NY', '42': 'PA', '09': 'CT', '10': 'DE'}
acs_combined['state'] = acs_combined['state'].map(state_map)

acs_final = acs_combined[['NAME', 'state', 'fips',
                          'pop_total', 'median_age', 'pop_elderly', 'avg_household_size',
                          'median_income', 'pop_below_poverty',
                          'labour_force', 'unemployed',
                          'total_commuters', 'public_transport_commuters',
                          'pop_white', 'pop_foreign_born', 'pop_bachelors_plus']]

print(f'ACS final: {acs_final.shape}, missing: {acs_final.isnull().sum().sum()}')
acs_final.to_csv(RAW_DIR / 'acs_demographics' / 'acs_multistate_2022.csv', index=False)
print(f'✅ Saved: acs_demographics/acs_multistate_2022.csv')
print(f'\nState breakdown:')
print(acs_final.groupby("state").size())

ACS final: (141, 16), missing: 0
✅ Saved: acs_demographics/acs_multistate_2022.csv

State breakdown:
state
CT     9
DE     3
NY    62
PA    67
dtype: int64


In [13]:
# Land area from Census Gazetteer
GAZ_URL = 'https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2022_Gazetteer/2022_Gaz_counties_national.zip'
import zipfile, io
with urllib.request.urlopen(GAZ_URL, timeout=30) as r:
    zdata = io.BytesIO(r.read())
with zipfile.ZipFile(zdata) as z:
    fname = [n for n in z.namelist() if n.endswith('.txt')][0]
    with z.open(fname) as f:
        gaz = pd.read_csv(f, sep='\t', encoding='latin-1')
gaz.columns = gaz.columns.str.strip()
gaz['fips'] = gaz['GEOID'].astype(str).str.zfill(5)
land = gaz[['fips', 'NAME', 'ALAND_SQMI']].rename(columns={'ALAND_SQMI': 'land_area_sqmi'})

target = set(acs_final['fips'])
land_target = land[land['fips'].isin(target)].copy()
land_target.to_csv(RAW_DIR / 'acs_demographics' / 'land_area_2022.csv', index=False)
print(f'✅ Land area saved: {len(land_target)} areas')

✅ Land area saved: 141 areas


---

## Summary

In [14]:
print('=== Acquired Data Summary ===\n')
expected = [
    ('ny_flu_panel.csv', 'NY flu panel (62 counties × 17 seasons)'),
    ('ny_demographics_tidy.csv', 'NY demographics snapshot (62 counties × 64 cols)'),
    ('ny_vaccination_2024.csv', 'NY weekly vaccination 2024-25'),
    ('pennsylvania/pa_flu_panel.csv', 'PA flu cumulative 2025-26 (67 counties)'),
    ('connecticut/ct_flu_panel.csv', 'CT flu panel (9 planning regions × 3 seasons)'),
    ('delaware/de_flu_panel.csv', 'DE flu panel (3 counties × 6 seasons)'),
    ('acs_demographics/acs_multistate_2022.csv', 'ACS 2022 demographics (NY+PA+CT+DE, 141 areas)'),
    ('acs_demographics/land_area_2022.csv', 'Land area for density calc (141 areas)'),
]
print(f'{"File":<55} {"Status":<10}  Description')
print('-' * 110)
for fname, desc in expected:
    fpath = RAW_DIR / fname
    status = f'✅ {fpath.stat().st_size / 1024:.1f} KB' if fpath.exists() else '❌ missing'
    print(f'{fname:<55} {status:<10}  {desc}')

print(f'\n📂 All files written to: {RAW_DIR}')
print(f'\n🎯 Cross-section size:')
print(f'   NY 2025-26 (62 counties)')
print(f' + PA 2025-26 (67 counties)')
print(f' + CT 2024-25 (9 planning regions)')
print(f' + DE 2024-25 (3 counties)')
print(f' = 141 areas for cross-sectional analysis')
print(f'\n   Plus NY temporal panel 2009-2026 (1,032 obs) for sensitivity analysis')
print(f'\n👉 Next: open 01_master_dataframe.ipynb to merge into the master cross-section CSV.')

=== Acquired Data Summary ===

File                                                    Status      Description
--------------------------------------------------------------------------------------------------------------
ny_flu_panel.csv                                        ✅ 67.4 KB   NY flu panel (62 counties × 17 seasons)
ny_demographics_tidy.csv                                ✅ 19.7 KB   NY demographics snapshot (62 counties × 64 cols)
ny_vaccination_2024.csv                                 ✅ 107.7 KB  NY weekly vaccination 2024-25
pennsylvania/pa_flu_panel.csv                           ✅ 3.6 KB    PA flu cumulative 2025-26 (67 counties)
connecticut/ct_flu_panel.csv                            ✅ 2.1 KB    CT flu panel (9 planning regions × 3 seasons)
delaware/de_flu_panel.csv                               ✅ 1.2 KB    DE flu panel (3 counties × 6 seasons)
acs_demographics/acs_multistate_2022.csv                ✅ 15.7 KB   ACS 2022 demographics (NY+PA+CT+DE, 141 areas)
acs_demograp

---

## Appendix — States We Investigated and Skipped

After exhaustive testing (~25 states checked via direct API calls), the following are confirmed not viable:

| State | Issue |
|---|---|
| **California** | Flu data published only at HHS region level. The CHHS infectious disease county dataset explicitly excludes influenza. |
| **Florida** | Florida law does not require individual flu case reporting except for novel strains and pediatric deaths. |
| **Virginia** | "VDH PUD Influenza Labs" is state-level only. ED visit dataset has empty `County FIPS` column. Health districts (41), not counties. |
| **Colorado** | CDPHE current data has only `level_=ALL` (statewide). 2015-2019 county aggregate exists but is single static snapshot. |
| **Rhode Island** | Cloudflare-blocked open data portal. Cannot programmatically access. |
| **New Jersey** | Zero flu datasets at data.nj.gov. NJSHAD blocks bot access. |
| **Maryland** | Only vaccination data (`3abd-txk7`), no case counts. |
| **Iowa, Maine, Vermont, NH, NM, NE, KS, OK, DC, UT** | No flu datasets at any open data portal. |
| **Texas, Ohio, Illinois, Michigan, Minnesota, Massachusetts, Wisconsin** | State PDF-only or HHS-region-only surveillance. |
| **Louisiana, Missouri** | Tableau dashboards without programmatic access. |

## What this teaches us

US flu surveillance is **state-controlled and inconsistently published**. Only ~8% of states publish county-level lab-confirmed flu data publicly via API:
- **NY, PA, CT, DE** have public county-level data (we use all four)
- **CO** has it but only as a single 2015-2019 snapshot (not useful for time series)
- All others: state-level, region-level, PDF-only, or behind anti-bot walls

This is why the project pivoted from "reproduce a state-of-the-art forecasting model" to "build a defensible cross-sectional vulnerability model on the available data".